# Notebook 06 — Modelos Teóricos Clássicos (Item 5 da Parte III)

**Aluno:** Vitor Renato Santos dos Santos | **Matrícula:** 222217109 — UFBA
**Dataset:** ego-Gplus (G2) — SNAP | **Grafo:** G2 ($n = 107.614$, $m = 12.238.285$)

Este notebook compara o grafo real com três modelos teóricos clássicos:
- **Erdős–Rényi (ER):** G(n, m) com mesmo n e m do real.
- **Barabási–Albert (BA):** Anexação preferencial com m_attach = 114.
- **Watts–Strogatz (WS):** Anel regular com k = 228 vizinhos e religação β ∈ {0.01, 0.1, 0.5}.

Todos os geradores são manuais (Numba sobre CSR), sem uso de igraph/networkx para geração.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESULTS_DIR = "results"
FIGURES_DIR = "figures"

## 1. Resultados Pré-Computados

In [ ]:
with open(os.path.join(RESULTS_DIR, "parte3_extra_modelos.json"), "r", encoding="utf-8") as f:
    res = json.load(f)

df = pd.read_csv(os.path.join(RESULTS_DIR, "parte3_extra_tabela_comparativa.csv"))
print(df.to_string(index=False))

## 2. Validação com igraph (n = 2.000)

In [ ]:
val = res["validacao_igraph_n2000"]
for model, info in val.items():
    status = "✓" if info["match_m"] and info["match_C"] else "✗"
    print(f"{model}: m_manual={info['m_manual']:,} vs igraph={info['m_igraph']:,} ({status})")
    print(f"  C_manual={info['C_manual']:.6f} vs igraph={info['C_igraph']:.6f}")

## 3. CCDF de Graus (Real vs. Modelos)

In [ ]:
fig_ccdf = os.path.join(FIGURES_DIR, "parte3_extra_ccdf_graus.png")
if os.path.exists(fig_ccdf):
    from IPython.display import Image, display
    display(Image(filename=fig_ccdf))
else:
    print(f"Figura não encontrada: {fig_ccdf}")

**Análise:**
- O grafo real (preto) apresenta cauda pesada que se estende até k ≈ 20.000.
- **ER** (vermelho) tem distribuição de Poisson com decaimento exponencial rápido — sem hubs.
- **BA** (verde) produz cauda power-law, mais próxima do real, mas sem a curvatura observada.
- **WS** (roxo) é praticamente regular (todos os nós ≈ k = 228) — sem heterogeneidade.

## 4. L e C por Modelo

In [ ]:
fig_bars = os.path.join(FIGURES_DIR, "parte3_extra_L_C_barras.png")
if os.path.exists(fig_bars):
    from IPython.display import Image, display
    display(Image(filename=fig_bars))
else:
    print(f"Figura não encontrada: {fig_bars}")

**Análise de L (Average Path Length):**
- **ER** e **BA** produzem L comparável ao real (~2.6–2.9) → propriedade de mundo pequeno.
- **WS com β baixo** (0.01) tem L mais alto (rede mais "regular", poucos atalhos).
- **WS com β alto** (0.5) se aproxima de L do ER (muitos atalhos).

**Análise de C (Clusterização):**
- **Real** tem C ≈ 0.49, extremamente alto.
- **ER** tem C ≈ ⟨k⟩/n ≈ 0.002 (200× menor) — falha total.
- **BA** tem C baixo (~0.01) — falha significativa.
- **WS β=0.01** preserva C alto (~0.6) → sucesso aqui.
- **WS β=0.5** reduz C consideravelmente.

## 5. Conclusão: Qual Modelo Reproduz Qual Propriedade?

| Propriedade | Real | ER | BA | WS (β baixo) | WS (β alto) |
|-------------|------|----|----|---------------|-------------|
| L pequeno | 2.86 | ✓ | ✓ | Parcial | ✓ |
| C alto | 0.49 | ✗ | ✗ | ✓ | Parcial |
| Cauda pesada | Sim | ✗ | ✓ | ✗ | ✗ |

**Nenhum modelo isolado reproduz todas as propriedades do grafo real.**

- **ER** captura L mas falha em C e na cauda.
- **BA** captura L e cauda pesada, mas C é 50× menor que o real.
- **WS** (β baixo) captura C alto e L moderado, mas graus são homogêneos.

O grafo G2 (ego-Gplus) combina L curto, C muito alto e heterogeneidade de graus
(embora não power-law pura, como mostrado no Item 2). Modelos híbridos como
Holme-Kim (BA + fechamento triádico) ou modelos de comunidades seriam necessários
para reproduzir simultaneamente essas três propriedades.

## 6. Parâmetros e Tempos de Execução

In [ ]:
params = res["parametros"]
print(f"n = {params['n']:,}")
print(f"m_real = {params['m_real']:,}")
print(f"m_attach (BA) = {params['m_attach_BA']}")
print(f"k (WS) = {params['k_WS']}")
print(f"betas (WS) = {params['betas_WS']}")
print(f"seed = {params['seed']}")
print(f"k_APL = {params['k_APL']}")

for name in ["ER", "BA", "WS_b0.01", "WS_b0.1", "WS_b0.5"]:
    if name in res["modelos"]:
        info = res["modelos"][name]
        print(f"\n{name}:")
        print(f"  Geração: {info.get('t_gen_s', '?')}s")
        print(f"  APL: {info.get('t_apl_s', '?')}s")
        print(f"  Clustering: {info.get('t_clust_s', '?')}s")
        print(f"  Memória CSR: {info.get('mem_csr_mb', '?')} MB")

## 7. Limitações

1. **Uma única realização por modelo** (seed=42); a variância inter-sementes não é avaliada.
2. **Parâmetros calibrados para ⟨k⟩:** m_attach=114 (BA) e k=228 (WS) são escolhidos
   para igualar o grau médio do real, mas outros valores poderiam alterar os resultados.
3. **APL por amostragem** (k=1.000 fontes): erro amostral ≈ 1–2%.
4. **γ Clauset em ER/WS** não faz sentido teórico (distribuição Poisson/regular).
5. **WS com n grande e k=228** é custoso para gerar; tempo dominado pela hash table de dedup.